# Probability in machine learning

Section 03 uses squared error for regression and cross-entropy for classification. They are not arbitrary. Each one is the negative log of a probability model. This notebook shows where.

Order: intuition, maths, idea, from scratch, library, what breaks.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)

from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import log_loss

## 1. Intuition

Pick a probability model for how the data was produced. Maximum likelihood estimation (MLE) chooses the parameters that make the observed data most probable. A loss function is just that goal turned upside down: minimise the negative log-likelihood.

So choosing a loss is choosing an assumption about noise.

## 2. Maths

For independent data, the likelihood is a product, and the log turns it into a sum:

$$\hat\theta=\arg\max_\theta\prod_i p(y_i\mid x_i,\theta)=\arg\min_\theta\left[-\sum_i\log p(y_i\mid x_i,\theta)\right]$$

**Gaussian noise gives squared error.** If $y=f(x)+\varepsilon$ with $\varepsilon\sim N(0,\sigma^2)$:

$$-\log p(y\mid x)=\frac{(y-f(x))^2}{2\sigma^2}+\log(\sigma\sqrt{2\pi})$$

Summed over the data and with $\sigma$ fixed, this is the sum of squared errors times a constant.

**Bernoulli gives cross-entropy.** If $y\in\{0,1\}$ and $P(y=1)=\hat p$:

$$-\log p(y)=-\left[y\log\hat p+(1-y)\log(1-\hat p)\right]$$

**Categorical gives softmax cross-entropy.** With $K$ classes, $\hat p=\mathrm{softmax}(z)$, $\hat p_k=e^{z_k}/\sum_j e^{z_j}$ and the loss is $-\log\hat p_{y}$.

**A prior gives regularisation.** MAP estimation maximises likelihood times prior, so it adds $-\log p(\theta)$ to the loss. A Gaussian prior $\theta\sim N(0,\tau^2)$ adds $\|\theta\|^2/(2\tau^2)$, which is an L2 penalty.

## 3. Gaussian noise gives squared error

Generate data from a line with Gaussian noise. Minimise the negative log-likelihood numerically and compare with the least-squares solution from section 01.

In [ ]:
n = 200
x = rng.uniform(-3, 3, n)
y = 2.0 * x + 1.0 + rng.normal(0, 1.5, n)
A = np.column_stack([x, np.ones(n)])


def gaussian_nll(params: np.ndarray) -> float:
    w, b, log_sigma = params
    sigma = np.exp(log_sigma)
    resid = y - (w * x + b)
    return float(np.sum(resid**2 / (2 * sigma**2) + np.log(sigma * np.sqrt(2 * np.pi))))


fit = minimize(gaussian_nll, x0=[0.0, 0.0, 0.0])
ls = np.linalg.lstsq(A, y, rcond=None)[0]
print("MLE (w, b)    :", fit.x[:2])
print("least squares :", ls)
print("MLE sigma     :", np.exp(fit.x[2]), " residual sd:", (y - A @ ls).std())

The MLE slope and intercept match least squares, and the MLE noise level is the residual standard deviation. Heavier-tailed noise changes the answer. Laplace noise gives absolute error, which is estimated by the median rather than the mean.

In [ ]:
sample = np.concatenate([rng.normal(10, 1, 99), [1000.0]])  # one wild outlier
print("mean (Gaussian MLE)  :", sample.mean())
print("median (Laplace MLE) :", np.median(sample))

## 4. Bernoulli gives cross-entropy

Fit a logistic model by minimising the Bernoulli negative log-likelihood. Check the value against sklearn's `log_loss` and the weights against `LogisticRegression` with no penalty.

In [ ]:
n = 500
X = rng.normal(0, 1, (n, 2))
true_w = np.array([1.5, -2.0])
p_true = 1 / (1 + np.exp(-(X @ true_w + 0.5)))
y_bin = (rng.random(n) < p_true).astype(int)


def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-z))


def bce(params: np.ndarray) -> float:
    p = np.clip(sigmoid(X @ params[:2] + params[2]), 1e-12, 1 - 1e-12)
    return float(-np.mean(y_bin * np.log(p) + (1 - y_bin) * np.log(1 - p)))


fit = minimize(bce, x0=np.zeros(3))
clf = LogisticRegression(penalty=None).fit(X, y_bin)
print("from scratch:", fit.x)
print("sklearn     :", np.append(clf.coef_[0], clf.intercept_))
print("loss (mine, sklearn log_loss):", bce(fit.x), log_loss(y_bin, clf.predict_proba(X)))

## 5. Softmax is a categorical distribution

Softmax turns $K$ raw scores into $K$ probabilities that sum to 1. The loss is the negative log-probability of the true class. Subtract the max score first so the exponentials cannot overflow.

In [ ]:
def softmax(z: np.ndarray) -> np.ndarray:
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


z = np.array([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3], [1000.0, 1001.0, 999.0]])
labels = np.array([0, 1, 1])
p = softmax(z)
print(p)
print("rows sum to:", p.sum(axis=1))
ce = -np.log(p[np.arange(3), labels]).mean()
print("cross-entropy:", ce, " sklearn:", log_loss(labels, p, labels=[0, 1, 2]))

With two classes softmax reduces to the sigmoid, so binary cross-entropy is a special case.

## 6. MAP: a Gaussian prior is L2

Put $w\sim N(0,\tau^2)$ on the weights, with Gaussian noise of variance $\sigma^2$. Maximising likelihood times prior gives ridge regression with $\lambda=\sigma^2/\tau^2$:

$$\hat w=(X^\top X+\lambda I)^{-1}X^\top y$$

Check that against sklearn's `Ridge`, on a small noisy problem with many features where plain least squares overfits.

In [ ]:
n, d = 30, 20
X = rng.normal(0, 1, (n, d))
w_true = np.zeros(d)
w_true[:3] = [3.0, -2.0, 1.0]
sigma = 1.0
y_reg = X @ w_true + rng.normal(0, sigma, n)

tau = 1.0
lam = sigma**2 / tau**2
w_map = np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y_reg)
w_ridge = Ridge(alpha=lam, fit_intercept=False).fit(X, y_reg).coef_
w_mle = np.linalg.lstsq(X, y_reg, rcond=None)[0]

print("MAP matches Ridge:", np.allclose(w_map, w_ridge))
print("weight error, MLE:", np.linalg.norm(w_mle - w_true))
print("weight error, MAP:", np.linalg.norm(w_map - w_true))

A tighter prior (small $\tau$) means a larger $\lambda$ and smaller weights. As $\tau\to\infty$ the prior goes flat and MAP becomes MLE. Same relationship as in the Beta-Binomial notebook: the prior is a set of pseudo-observations that fades as data grows.

In [ ]:
for tau in (0.1, 0.5, 1.0, 5.0, 100.0):
    w = np.linalg.solve(X.T @ X + (sigma**2 / tau**2) * np.eye(d), X.T @ y_reg)
    print(f"tau={tau:>6}: |w|={np.linalg.norm(w):.2f}  error={np.linalg.norm(w - w_true):.2f}")

## 7. What breaks

In [ ]:
# 1. Perfectly separable data: the MLE pushes the weight towards infinity
xs = np.array([-2.0, -1.0, 1.0, 2.0])
ys = np.array([0, 0, 1, 1])


def nll_1d(w: float, prior_tau: float | None = None) -> float:
    p = np.clip(sigmoid(w * xs), 1e-12, 1 - 1e-12)
    loss = -np.sum(ys * np.log(p) + (1 - ys) * np.log(1 - p))
    if prior_tau is not None:
        loss += w**2 / (2 * prior_tau**2)
    return float(loss)


for w in (1, 5, 10, 20):
    print(f"w={w:>2}: NLL={nll_1d(w):.6f}  with prior: {nll_1d(w, 2.0):.3f}")

# 2. Wrong noise model: squared error on data with outliers
x_o = np.linspace(0, 10, 50)
y_o = 2 * x_o + rng.normal(0, 0.5, 50)
y_o[-1] += 100
print("slope, clean fit    :", np.polyfit(x_o[:-1], y_o[:-1], 1)[0])
print("slope, with outlier :", np.polyfit(x_o, y_o, 1)[0])

Without a prior the loss keeps falling as $w$ grows, so the optimiser never stops. With the prior it has a minimum. One outlier in 50 points drags the least-squares slope well away from 2, because a Gaussian says a 100-unit error is essentially impossible and the fit bends to explain it.

Section 03 starts from here: pick the noise model, get the loss, fit the parameters.